# M3L4 E03 — Debugging con traces
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitás saber antes

| Módulo | Concepto | Por qué lo necesitás acá |
|---|---|---|
| M3L4 E00 | Trace, Span, jerarquía | Los 5 casos de falla usan la estructura trace-span que viste en E00 |
| M3L4 E01-E02 | MiniTracer, duration_ms, metadata | Los campos que revisa `diagnose_trace()` son los mismos que construiste |
| M3L1 | Tool contracts | Un span con `output: None` es como una tool que no devuelve contrato |
| M3L2 | Agentes con estado | El loop de agentes (Caso 4) ocurre cuando el estado no avanza |
| Python | `dict.get()`, `Counter` de `collections` | Los usas para inspeccionar traces de forma robusta |

Si no entendés la estructura de un trace, repasá E00 antes de continuar.

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Diagnóstico** | Proceso de identificar automáticamente el tipo de falla en una traza | `diagnose_trace()` retorna `{'problem': 'misclassification', ...}` |
| **Misclassification** | El router clasifica el intent incorrectamente | `metadata.expected_intent != metadata.actual_intent` |
| **Retrieval vacío** | La búsqueda de documentos no encuentra resultados | Span de retrieval con `output['count'] == 0` o `output['documents'] == []` |
| **Latencia alta** | Un paso del sistema tarda más de lo esperado | `duration_ms > 3000` en algún span |
| **Loop de agentes** | El mismo agente aparece ejecutándose repetidamente | Mismo `name` de span aparece más de 2 veces |
| **Error silencioso** | Un span termina con `output: None` sin marcar error | Span con `output == None` |
| **suggested_fix** | Mensaje de acción correctiva sugerida | `'Revisar reglas de routing para diferenciar HR de Finance'` |

---

## Cómo encaja esto en un sistema de agentes

```
E02: Sistema multi-agente con tracing
    |  Cada request produce un trace con spans
    v
E03: Diagnosticar automáticamente fallas en esos traces
    |  diagnose_trace(trace) -> problem, details, suggested_fix
    v
E04: Golden datasets para medir accuracy del router
    |  Prevención: detectar misclassification antes de producirla
    v
E11-E12: Alertas, dashboards y mejora continua
    |  Monitoreo automático en producción
```

**Objetivo del ejercicio:** aprender a leer trazas para detectar y diagnosticar fallas típicas en sistemas de agentes.

## Instalación e imports

Este ejercicio solo necesita la biblioteca estándar de Python:

| Import | Qué hace | Por qué lo necesitamos |
|---|---|---|
| `import json` | (Opcional) Formatear traces como JSON para inspección visual | Para debuggear traces manualmente si es necesario |

`diagnose_trace()` no requiere imports adicionales porque opera sobre diccionarios nativos de Python.

```python
import json
```

In [ ]:
import json

## Caso 1 — Misclassification

**Problema:** el router envió la consulta al agente incorrecto.

**Síntoma en la traza:** `metadata.expected_intent != metadata.actual_intent`

**Escenario:** un usuario pregunta por su factura (`expected_intent: 'finance'`), pero el sistema lo clasifica como IT y responde con sugerencias técnicas irrelevantes.

```
metadata: {
    'expected_intent': 'finance',   # lo que DEBERÍA ser
    'actual_intent': 'it'           # lo que el router DIO
}
```

In [ ]:
trace_misclassification = {
    'trace_name': 'support-request',
    'input': {'query': 'No puedo ver mi factura'},
    'metadata': {
        'expected_intent': 'finance',
        'actual_intent': 'it'
    },
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'intent': 'it'},
            'duration_ms': 140
        },
        {
            'name': 'it-agent',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'response': 'Probá reiniciar la app.'},
            'duration_ms': 900
        }
    ],
    'output': {'final_response': 'Probá reiniciar la app.'}
}

trace_misclassification

## Caso 2 — Retrieval vacío

**Problema:** el sistema no encontró documentos relevantes en la base de conocimiento.

**Síntoma en la traza:** span de retrieval con `output['count'] == 0` o `output['documents'] == []`

**Escenario:** un usuario pregunta por política de licencia por maternidad, el router detecta HR correctamente, pero el retrieval devuelve 0 documentos porque no hay data indexada sobre ese tema.

```
retrieval span: {
    'output': {
        'documents': [],    # no se encontró nada
        'count': 0          # cero resultados
    }
}
```

In [ ]:
trace_empty_retrieval = {
    'trace_name': 'rag-support-request',
    'input': {'query': 'Cuál es la política de licencia por maternidad?'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Cuál es la política de licencia por maternidad?'},
            'output': {'intent': 'hr'},
            'duration_ms': 110
        },
        {
            'name': 'retrieval',
            'input': {'query': 'Cuál es la política de licencia por maternidad?'},
            'output': {'documents': [], 'count': 0},
            'duration_ms': 250
        },
        {
            'name': 'hr-agent',
            'input': {'query': 'Cuál es la política de licencia por maternidad?', 'context': ''},
            'output': {'response': 'No tengo información disponible sobre ese tema.'},
            'duration_ms': 820
        }
    ],
    'output': {'final_response': 'No tengo información disponible sobre ese tema.'}
}

trace_empty_retrieval

## Caso 3 — Latencia alta

**Problema:** un paso del sistema tarda mucho más de lo esperado.

**Síntoma en la traza:** `duration_ms > 3000` en algún span

**Escenario:** el agente HR tardó 5.8 segundos en responder. El umbral de alerta se define en 3 segundos. Posibles causas: timeout de API, LLM lento, query compleja.

```
hr-agent span: {
    'duration_ms': 5800   # > 3000 -> ALERTA
}
```

In [ ]:
trace_high_latency = {
    'trace_name': 'support-request',
    'input': {'query': 'Cómo solicito vacaciones?'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Cómo solicito vacaciones?'},
            'output': {'intent': 'hr'},
            'duration_ms': 95
        },
        {
            'name': 'hr-agent',
            'input': {'query': 'Cómo solicito vacaciones?'},
            'output': {'response': 'Para solicitar vacaciones ingresa al portal de RRHH.'},
            'duration_ms': 5800
        }
    ],
    'output': {'final_response': 'Para solicitar vacaciones ingresa al portal de RRHH.'}
}

trace_high_latency

## Caso 4 — Loop de agentes

**Problema:** dos o más agentes se pasan la solicitud entre sí sin llegar a una respuesta final.

**Síntoma en la traza:** el mismo `name` de span aparece más de 2 veces

**Escenario:** un usuario tiene un problema mixto (laptop lenta + factura). IT-agent deriva a Finance, Finance deriva de vuelta a IT, y así sucesivamente.

```
supervisor -> it-agent -> finance-agent -> it-agent -> finance-agent
                                                     ^--- loop detectado: 'it-agent' aparece 2 veces
```

In [ ]:
trace_loop = {
    'trace_name': 'support-request',
    'input': {'query': 'Mi laptop está lenta y tengo un problema con mi factura'},
    'spans': [
        {'name': 'supervisor', 'output': {'next': 'it-agent'}, 'duration_ms': 130},
        {'name': 'it-agent', 'output': {'response': 'Revisá tu conexión', 'handoff': 'finance-agent'}, 'duration_ms': 720},
        {'name': 'finance-agent', 'output': {'response': 'Revisá tu factura', 'handoff': 'it-agent'}, 'duration_ms': 680},
        {'name': 'it-agent', 'output': {'response': 'Revisá tu conexión', 'handoff': 'finance-agent'}, 'duration_ms': 710},
        {'name': 'finance-agent', 'output': {'response': 'Revisá tu factura', 'handoff': None}, 'duration_ms': 690}
    ],
    'output': {'final_response': 'Revisá tu factura'}
}

trace_loop

## Caso 5 — Error silencioso

**Problema:** un span termina sin output y sin marcar error.

**Síntoma en la traza:** span con `output == None`

**Escenario:** el legal-agent ejecuta pero no produce respuesta. La duración de 15ms sugiere que falló inmediatamente (un LLM normal tardaría 500-2000ms). El sistema no registró ningún error.

```
legal-agent span: {
    'output': None,       # debería ser un string
    'duration_ms': 15     # anormalmente bajo -> sugiere crash temprano
}
```

In [ ]:
trace_silent_error = {
    'trace_name': 'support-request',
    'input': {'query': 'Necesito ver mi contrato'},
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'Necesito ver mi contrato'},
            'output': {'intent': 'legal'},
            'duration_ms': 100
        },
        {
            'name': 'legal-agent',
            'input': {'query': 'Necesito ver mi contrato'},
            'output': None,
            'duration_ms': 15
        }
    ],
    'output': {'final_response': None}
}

trace_silent_error

## TODO — Función `diagnose_trace`

Implementa la función que detecta automáticamente el tipo de problema en una traza.

### Orden de detección

1. **Misclassification** -> revisar `metadata`
2. **Retrieval vacío** -> revisar spans con 'retrieval' en el nombre
3. **Latencia alta** -> revisar `duration_ms` de cada span
4. **Loop de agentes** -> contar frecuencias de nombres de spans
5. **Error silencioso** -> revisar spans con `output == None`

### Formato de retorno

```python
{
    'problem': 'misclassification',     # o 'empty_retrieval', 'high_latency', 'agent_loop', 'silent_error', 'none'
    'details': 'expected_intent=finance vs actual_intent=it',  # explicación del problema
    'suggested_fix': 'Revisar reglas de routing...',            # acción correctiva
}
```

In [ ]:
def diagnose_trace(trace: dict) -> dict:
    """
    Analiza una traza y devuelve el diagnóstico del problema encontrado.

    Debe detectar (en orden):
    1. misclassification: metadata.expected_intent != metadata.actual_intent
    2. empty_retrieval: algún span de retrieval con output.count == 0
    3. high_latency: algún span con duration_ms > 3000
    4. agent_loop: algún nodo aparece más de 2 veces en los spans
    5. silent_error: algún span con output == None

    Returns:
        dict con 'problem', 'details' y 'suggested_fix'
    """
    # TODO 1: detectar misclassification
    # Pista: comparar metadata.get('expected_intent') con metadata.get('actual_intent')

    # TODO 2: detectar retrieval vacío
    # Pista: buscar span con 'retrieval' en el nombre y output.get('count') == 0

    # TODO 3: detectar latencia alta
    # Pista: iterar spans y verificar duration_ms > 3000

    # TODO 4: detectar loop de agentes
    # Pista: Counter de span names, buscar si alguno aparece > 2 veces

    # TODO 5: detectar error silencioso
    # Pista: buscar span con output == None

    return {'problem': 'none', 'details': 'No se detectaron problemas.', 'suggested_fix': None}

print('Función definida.')

## Ejecutar el diagnóstico en todos los casos

Probamos `diagnose_trace()` contra cada uno de los 5 casos y vemos los resultados.

In [ ]:
cases = [
    ('Misclassification', trace_misclassification),
    ('Retrieval vacío',   trace_empty_retrieval),
    ('Latencia alta',     trace_high_latency),
    ('Loop de agentes',  trace_loop),
    ('Error silencioso',  trace_silent_error)
]

for name, trace in cases:
    result = diagnose_trace(trace)
    print(f'--- {name} ---')
    print(f"  Problema:   {result.get('problem')}")
    print(f"  Detalles:   {result.get('details')}")
    print(f"  Fix sugerido: {result.get('suggested_fix')}")
    print()

In [ ]:
assert diagnose_trace(trace_misclassification)['problem'] == 'misclassification'
assert diagnose_trace(trace_empty_retrieval)['problem'] == 'empty_retrieval'
assert diagnose_trace(trace_high_latency)['problem'] == 'high_latency'
assert diagnose_trace(trace_loop)['problem'] == 'agent_loop'
assert diagnose_trace(trace_silent_error)['problem'] == 'silent_error'
print('Checks E03 OK')

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| No revisar `metadata` primero | Empezar por otros chequeos y no detectar misclassification | El problema reportado no es el más grave |
| Usar `trace['metadata']` sin `.get()` | La clave puede no existir y lanza `KeyError` | Usar `trace.get('metadata', {})` como safe access |
| Confundir loop con repetición normal | Un agente puede aparecer 2 veces si el usuario consulta dos temas | Definir umbral: > 2 repeticiones = loop |
| No detectar error silencioso por `output=None` | Revisar `output == None` en vez de `output is None` | La duración anormalmente baja es otra pista |
| Retornar fix genérico | Sugerir lo mismo para todo tipo de falla | Cada problema necesita su propia acción correctiva |

## Síntesis

### Los 5 patrones de falla

| Tipo | Cómo se detecta | Acción correctiva típica |
|---|---|---|
| Misclassification | `expected_intent != actual_intent` | Agregar reglas al router o mejorar prompts |
| Retrieval vacío | Span con `count == 0` | Indexar más documentos o mejorar chunking |
| Latencia alta | `duration_ms > 3000` | Optimizar el paso lento o aumentar timeout |
| Loop de agentes | Mismo nombre > 2 veces | Agregar max_handoffs o mejorar lógica de delegación |
| Error silencioso | `output == None` | Agregar try/except y logging de errores |

### Relación con otros ejercicios

| Ejercicio | Conexión con E03 |
|---|---|
| **E04** | Golden datasets para prevenir misclassification antes de producirla |
| **E05** | Router v1 vs v2: comparar accuracy entre versiones |
| **E11** | Alertas automáticas cuando se detectan estos patrones en producción |
| **E12** | Ciclo de mejora: diagnosticar -> corregir -> medir de nuevo |